In [ ]:
# %% [markdown]
# # Airline Satisfaction EDA
# 尺度ごとに 平均・分散 / Q-Q plot / 相関係数 / PCA / t-SNE を計算する
# (Kaggle Notebook や VS Code では "# %%" ごとにセルとして実行できます)

# %%
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

# ===== 設定 =====
TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e10/train.csv"
TARGET = "satisfaction"
SAMPLE_N_PCA = 20000    # PCA 散布図の描画用サンプル数
SAMPLE_N_TSNE = 10000   # t-SNE 用サンプル数(多いと非常に遅い)
SEED = 42

# ===== 尺度ごとの列定義 =====
# 量的変数(比率尺度)
QUANT_COLS = [
    "Age", "Flight Distance",
    "Departure Delay in Minutes", "Arrival Delay in Minutes",
]
# 質的変数のうち、順序尺度(サービス評価 0〜5)
ORDINAL_COLS = [
    "Inflight wifi service", "Departure/Arrival time convenient",
    "Ease of Online booking", "Gate location", "Food and drink",
    "Online boarding", "Seat comfort", "Inflight entertainment",
    "On-board service", "Leg room service", "Baggage handling",
    "Checkin service", "Cleanliness",
]
# 質的変数のうち、名義尺度
NOMINAL_COLS = ["Gender", "Customer Type", "Type of Travel", "Class"]
DELAY_COLS = ["Departure Delay in Minutes", "Arrival Delay in Minutes"]

# %%
# ===== データ読み込み =====
df = pd.read_csv(TRAIN_PATH)

# ターゲットが文字列なら 0/1 に変換
if not pd.api.types.is_numeric_dtype(df[TARGET]):
    df[TARGET] = (df[TARGET] == "satisfied").astype(int)

print("shape:", df.shape)
print("\n欠損数:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nターゲット比率:")
print(df[TARGET].value_counts(normalize=True))

# PCA / t-SNE 用に欠損を補完したコピー
# (到着遅延の欠損は出発遅延で補完し、残りは中央値)
df_f = df.copy()
if "Arrival Delay in Minutes" in df_f:
    df_f["Arrival Delay in Minutes"] = df_f["Arrival Delay in Minutes"].fillna(
        df_f["Departure Delay in Minutes"]
    )
for c in QUANT_COLS + ORDINAL_COLS:
    df_f[c] = df_f[c].fillna(df_f[c].median())

# %% [markdown]
# ## 1. 平均・分散などの要約統計

# %%
def describe_quant(data, cols):
    """量的変数: 平均・分散・標準偏差・歪度・尖度・分位点"""
    d = data[cols]
    out = pd.DataFrame({
        "mean": d.mean(),
        "var": d.var(),
        "std": d.std(),
        "skew": d.skew(),
        "kurtosis": d.kurt(),
        "min": d.min(),
        "q25": d.quantile(0.25),
        "median": d.median(),
        "q75": d.quantile(0.75),
        "max": d.max(),
        "n_missing": d.isna().sum(),
    })
    return out


def describe_ordinal(data, cols):
    """順序尺度: 平均・分散に加えて中央値・最頻値・0(該当なし)の割合"""
    d = data[cols]
    out = pd.DataFrame({
        "mean": d.mean(),
        "var": d.var(),
        "median": d.median(),
        "mode": d.mode().iloc[0],
        "zero_ratio": (d == 0).mean(),
        "n_unique": d.nunique(),
    })
    return out


def describe_nominal(data, cols, target):
    """名義尺度: カテゴリごとの度数・割合・満足率"""
    tables = {}
    for c in cols:
        t = data.groupby(c)[target].agg(count="size", satisfied_rate="mean")
        t["ratio"] = t["count"] / t["count"].sum()
        tables[c] = t[["count", "ratio", "satisfied_rate"]]
    return tables


print("=== 量的変数 ===")
display(describe_quant(df, QUANT_COLS).round(3))

print("=== 順序尺度(サービス評価) ===")
display(describe_ordinal(df, ORDINAL_COLS).round(3))

print("=== 順序尺度: 評価値ごとの度数分布 ===")
display(df[ORDINAL_COLS].apply(lambda s: s.value_counts(normalize=True)).T.round(3))

print("=== 名義尺度 ===")
for c, t in describe_nominal(df, NOMINAL_COLS, TARGET).items():
    print(f"\n--- {c} ---")
    display(t.round(3))

# %% [markdown]
# ## 2. Q-Q plot(量的変数のみ)
# 遅延は 0 に大きく偏るため、log1p 変換後も表示する

# %%
qq_targets = [(c, df[c].dropna(), "raw") for c in QUANT_COLS]
qq_targets += [(c, np.log1p(df[c].dropna()), "log1p") for c in DELAY_COLS]

n = len(qq_targets)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.ravel()

rng = np.random.default_rng(SEED)
for ax, (c, s, kind) in zip(axes, qq_targets):
    # 点が多すぎると描画が重いのでサンプリング
    if len(s) > 50000:
        s = s.iloc[rng.choice(len(s), 50000, replace=False)]
    stats.probplot(s, dist="norm", plot=ax)
    ax.set_title(f"{c} ({kind})", fontsize=10)
for ax in axes[n:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

# %% [markdown]
# ## 3. 相関係数
# - 量的変数: Pearson と Spearman
# - 順序尺度: Spearman
# - 名義尺度: Cramér の V
# - ターゲットとの関係: 量的・順序は点双列相関 / Spearman、名義は Cramér の V

# %%
def plot_corr(mat, title, fmt=".2f", annot=True, figsize=(8, 6), vmin=-1):
    plt.figure(figsize=figsize)
    sns.heatmap(mat, annot=annot, fmt=fmt, cmap="coolwarm",
                vmin=vmin, vmax=1, square=True, cbar_kws={"shrink": 0.8},
                annot_kws={"size": 8})
    plt.title(title)
    plt.tight_layout()
    plt.show()


# --- 量的変数 ---
quant_t = QUANT_COLS + [TARGET]
plot_corr(df[quant_t].corr(method="pearson"), "Quantitative: Pearson")
plot_corr(df[quant_t].corr(method="spearman"), "Quantitative: Spearman")

# --- 順序尺度 ---
ord_t = ORDINAL_COLS + [TARGET]
plot_corr(df[ord_t].corr(method="spearman"), "Ordinal (ratings): Spearman",
          figsize=(12, 10))


# --- 名義尺度: Cramér の V(バイアス補正版) ---
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum()
    r, k = ct.shape
    phi2 = chi2 / n
    phi2corr = max(0, phi2 - (k - 1) * (r - 1) / (n - 1))
    rcorr = r - (r - 1) ** 2 / (n - 1)
    kcorr = k - (k - 1) ** 2 / (n - 1)
    denom = min(kcorr - 1, rcorr - 1)
    return np.sqrt(phi2corr / denom) if denom > 0 else 0.0


nom_t = NOMINAL_COLS + [TARGET]
cv = pd.DataFrame(np.eye(len(nom_t)), index=nom_t, columns=nom_t)
for i, a in enumerate(nom_t):
    for b in nom_t[i + 1:]:
        v = cramers_v(df[a], df[b])
        cv.loc[a, b] = cv.loc[b, a] = v
plot_corr(cv, "Nominal: Cramer's V", vmin=0)

# --- ターゲットとの関係をまとめた一覧 ---
rows = []
for c in QUANT_COLS:
    rows.append((c, "quant", "pearson", df[[c, TARGET]].corr().iloc[0, 1]))
for c in ORDINAL_COLS:
    rows.append((c, "ordinal", "spearman",
                 df[[c, TARGET]].corr(method="spearman").iloc[0, 1]))
for c in NOMINAL_COLS:
    rows.append((c, "nominal", "cramers_v", cramers_v(df[c], df[TARGET])))
target_rel = (pd.DataFrame(rows, columns=["feature", "scale", "method", "value"])
              .assign(abs_value=lambda d: d["value"].abs())
              .sort_values("abs_value", ascending=False))
display(target_rel.round(3))

# %% [markdown]
# ## 4. PCA
# - 量的変数: 標準化して PCA
# - 順序尺度(評価13項目): 標準化して PCA
# - 名義尺度: one-hot 化して PCA(厳密には MCA が適切。近似として実施)

# %%
def run_pca(X, name, feature_names, y, sample_n=SAMPLE_N_PCA):
    Xs = StandardScaler().fit_transform(X)
    pca = PCA(random_state=SEED).fit(Xs)
    Z = pca.transform(Xs)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 寄与率と累積寄与率
    evr = pca.explained_variance_ratio_
    axes[0].bar(range(1, len(evr) + 1), evr, label="each")
    axes[0].plot(range(1, len(evr) + 1), np.cumsum(evr), "o-", c="C1",
                 label="cumulative")
    axes[0].set_xlabel("component")
    axes[0].set_title(f"{name}: explained variance")
    axes[0].legend()

    # 負荷量(上位 5 成分まで)
    k = min(5, len(evr))
    load = pd.DataFrame(pca.components_[:k].T, index=feature_names,
                        columns=[f"PC{i+1}" for i in range(k)])
    sns.heatmap(load, annot=True, fmt=".2f", cmap="coolwarm", center=0,
                ax=axes[1], annot_kws={"size": 7})
    axes[1].set_title(f"{name}: loadings")

    # PC1-PC2 散布図(ターゲットで色分け)
    idx = np.random.default_rng(SEED).choice(len(Z), min(sample_n, len(Z)),
                                             replace=False)
    sc = axes[2].scatter(Z[idx, 0], Z[idx, 1], c=y[idx], cmap="coolwarm",
                         s=3, alpha=0.4)
    axes[2].set_xlabel("PC1")
    axes[2].set_ylabel("PC2")
    axes[2].set_title(f"{name}: PC1 vs PC2 (color = target)")
    plt.colorbar(sc, ax=axes[2])

    plt.tight_layout()
    plt.show()

    print(f"[{name}] 累積寄与率:", np.round(np.cumsum(evr), 3))
    return pca, load


y = df_f[TARGET].values

pca_quant, load_quant = run_pca(df_f[QUANT_COLS].values, "Quantitative",
                                QUANT_COLS, y)
pca_ord, load_ord = run_pca(df_f[ORDINAL_COLS].values, "Ordinal ratings",
                            ORDINAL_COLS, y)

nom_oh = pd.get_dummies(df_f[NOMINAL_COLS].astype(str), drop_first=False,
                        dtype=float)
pca_nom, load_nom = run_pca(nom_oh.values, "Nominal (one-hot)",
                            nom_oh.columns.tolist(), y)

# %% [markdown]
# ## 5. t-SNE
# 計算が重いので SAMPLE_N_TSNE 行にサンプリングして実行
# - 量的変数のみ / 質的変数のみ(順序 + 名義 one-hot)/ 全変数 の 3 パターン

# %%
tsne_idx = np.random.default_rng(SEED).choice(
    len(df_f), min(SAMPLE_N_TSNE, len(df_f)), replace=False)
sub = df_f.iloc[tsne_idx]
y_sub = sub[TARGET].values

X_quant = StandardScaler().fit_transform(sub[QUANT_COLS])
X_qual = np.hstack([
    StandardScaler().fit_transform(sub[ORDINAL_COLS]),
    pd.get_dummies(sub[NOMINAL_COLS].astype(str), dtype=float).values,
])
X_all = np.hstack([X_quant, X_qual])

sets = {"Quantitative": X_quant, "Qualitative": X_qual, "All features": X_all}

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (name, X) in zip(axes, sets.items()):
    emb = TSNE(n_components=2, perplexity=30, init="pca",
               random_state=SEED).fit_transform(X)
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=y_sub, cmap="coolwarm",
                    s=3, alpha=0.5)
    ax.set_title(f"t-SNE: {name} (n={len(X)})")
    ax.set_xticks([])
    ax.set_yticks([])
plt.colorbar(sc, ax=axes, shrink=0.8, label=TARGET)
plt.show()